# Reading an H I observation file

Every observation this telescope records is one **HDF5** file in
`data/observations/`, named `YYYYMMDD_HHMMSS_<mode>.h5`. This notebook opens
one, prints what is in it, and plots it — a **tracked spectrum** as a spectrum,
a **drift scan** as its band-power curve through time.

You need only three packages: `h5py`, `numpy`, `matplotlib`. Nothing from the
scheduler is imported, so this runs anywhere the file does.

### What the file holds

One recording carries **two products** from the one radio stream:

| Product | Frequency axis | Spectra | Channels | Covers |
|---|---|---|---|---|
| **H I sub-band** | `frequency_hz` | `spectra_kelvin` *or* `spectra_linear` | ~845 | the 21 cm line, ~0.8 km/s per channel |
| **Continuum (wide)** | `frequency_hz_wide` | `spectra_wide_kelvin` *or* `spectra_wide_linear` | 1024 | the whole 8 MHz, the line excluded |

Per-record datasets, one row each: `timestamps` (Unix seconds, at the centre
of each integration), `integration_times` (seconds), `overflows` (samples the
SDR dropped that record — should be 0).

**The units are in the dataset name.** `spectra_kelvin` means the bandpass and
gain calibration were applied at write time; `spectra_linear` means they did
not apply to this tuning and the data are raw counts. Ask for the wrong one
and h5py raises `KeyError` rather than quietly handing you the other scale —
that guard is deliberate. `bandpass_correction` and `bandpass_valid` (and the
`_wide` versions) travel with the file so kelvin is exactly reversible back to
counts; the last cell shows how.

Everything else — the tuning, the calibration constants, where the dish
pointed, the site — is stored in **file attributes** (`hf.attrs`), listed in
full further down.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Choose a file

Recordings sort chronologically (the name leads with the timestamp), so
`FILE_INDEX = -1` is the most recent. Change it to open another.

In [ ]:
data_dir = Path('data') / 'observations'
folder = data_dir if data_dir.exists() else Path('.')
h5_files = sorted(folder.glob('*.h5'), key=lambda p: p.stat().st_mtime)

print(f'{len(h5_files)} recording(s) in {folder}:')
for i, f in enumerate(h5_files):
    print(f'  [{i}] {f.name}')

FILE_INDEX = -1  # -1 = most recent
h5_path = h5_files[FILE_INDEX]
print(f'\nSelected: {h5_path.name}')

## 2. Open it

A recording still being written must be opened with `swmr=True` (single
writer, many readers). A plain open hits the file lock, which is the right
answer for a file *not* written that way, so try plain first and fall back.

In [ ]:
try:
    hf = h5py.File(h5_path, 'r')
except OSError:
    hf = h5py.File(h5_path, 'r', swmr=True)
    print('(a recording in progress: reading what has arrived so far)')

attrs = dict(hf.attrs)

def product(prefix):
    """Return (freq_hz, spectra, units) for 'H I' or the wide product, or None.

    The dataset *name* is the units - never guess. A file in kelvin read as
    counts is wrong by the gain and offset by T_sys, and looks perfectly fine.
    """
    fkey = 'frequency_hz' if prefix == 'h1' else 'frequency_hz_wide'
    if fkey not in hf:
        return None
    base = 'spectra' if prefix == 'h1' else 'spectra_wide'
    if base + '_kelvin' in hf:
        return hf[fkey][:], hf[base + '_kelvin'][:], 'K'
    return hf[fkey][:], hf[base + '_linear'][:], 'counts'

h1 = product('h1')            # the 21 cm sub-band - always present
wide = product('wide')        # the continuum product - fixed-instrument files only

freq_hz, spectra, units = h1
freq_mhz = freq_hz / 1e6
timestamps = hf['timestamps'][:]
integ = hf['integration_times'][:]
n_rec, n_chan = spectra.shape
calibrated = units == 'K'

print(f'H I product : {n_rec} records x {n_chan} channels, in {units}')
print(f'              {freq_mhz[0]:.3f} - {freq_mhz[-1]:.3f} MHz')
if wide is not None:
    print(f'Continuum   : {wide[1].shape[1]} channels, {wide[0][0]/1e6:.3f} - {wide[0][-1]/1e6:.3f} MHz, in {wide[2]}')
else:
    print('Continuum   : none (recorded before the fixed instrument)')
print(f'Time span   : {(timestamps[-1] - timestamps[0]) / 60:.1f} min, total integration {integ.sum():.0f} s')
if 'overflows' in hf:
    dropped = int(hf['overflows'][:].sum())
    print(f'Overflows   : {dropped}' + ('  <-- samples were dropped!' if dropped else '  (clean)'))

## 3. The details

The tuning, the calibration in force, the pointing, and the observation
context — all from the file's attributes.

In [ ]:
def A(k, default='-'):
    return attrs.get(k, default)

print('=== Instrument ===')
print(f"  SDR              : {A('sdr_type')}")
print(f"  LO / centre      : {A('center_freq_hz', 0)/1e6:.4f} MHz")
print(f"  Sample rate      : {A('sample_rate_hz', 0)/1e6:.3f} Msps")
print(f"  Gain             : {A('gain_db')} dB")
print(f"  H I band         : {np.array(A('h1_band_hz', [0,0]))/1e6} MHz")
if 'continuum_band_hz' in attrs:
    print(f"  Continuum band   : {np.array(A('continuum_band_hz'))/1e6} MHz")

print('\n=== Calibration ===')
print(f"  Spectra units    : {A('spectra_units', units)}")
if calibrated:
    print(f"  Gain             : {A('applied_gain_counts_per_k'):.4g} counts/K")
    print(f"  System temp      : {A('applied_t_sys_k'):.1f} K")
    bad = (~hf['bandpass_valid'][:]).sum()
    if bad:
        print(f"  {bad} channels lay outside the bandpass template and are uncorrected")
else:
    print('  raw counts - no bandpass/gain applied to this tuning')

print('\n=== Site & beam ===')
print(f"  Location         : {A('site_lat_deg')}, {A('site_lon_deg')}  ({A('site_height_m')} m)")
print(f"  Beam FWHM        : {A('beam_fwhm_deg')} deg,  effective area {A('effective_area_m2', float('nan')):.2f} m^2")

if 'obs_name' in attrs:
    print('\n=== Observation ===')
    print(f"  Name             : {A('obs_name')}" + (f"  ({A('comment')})" if A('comment','') else ''))
    # track = the mount followed the sky; drift = it was parked and the sky
    # moved through the beam. This decides whether two records look at the
    # same piece of sky, so read it before averaging them together.
    print(f"  Mode             : {A('observation_mode')}  ({A('coord_system')})")
    cs = A('coord_system')
    c1 = A('coord1_deg',0)+A('coord1_min',0)/60+A('coord1_sec',0)/3600
    c2 = A('coord2_deg',0)+A('coord2_min',0)/60+A('coord2_sec',0)/3600
    label = {'object':'Object','radec':'RA/Dec','galactic':'Gal l/b','altaz':'Alt/Az'}.get(cs)
    if cs == 'object':
        print(f"  Target           : {A('object_name')}")
    elif label:
        print(f"  Target ({label:8s}): {c1:.3f}, {c2:.3f}")
    if 'drift_alt' in attrs:
        print(f"  Parked (drive)   : alt {A('drift_drive_alt')}, az {A('drift_drive_az')}")
        if 'drift_crossing_time' in attrs:
            print(f"  Beam crossing    : {A('drift_crossing_time')}  (offset {A('drift_crossing_offset_deg')} deg)")
    print(f"  Recorded (UTC)   : {A('created')}")
else:
    print('\n  (no scheduler metadata - the receiver was run by hand)')

## 4. A title from the metadata

In [ ]:
def make_title():
    parts = [str(attrs[k]) for k in ('obs_name',) if k in attrs]
    if A('coord_system') == 'object':
        parts.append(str(A('object_name','')).capitalize())
    if A('observation_mode','') :
        parts.append(str(A('observation_mode')))
    if A('calibrator', 0):
        parts.append('(CAL ON)')
    parts.append(str(A('created',''))[:19])
    return '  '.join(p for p in parts if p) or h5_path.name

title = make_title()
is_drift = str(A('observation_mode','')).lower() == 'drift'
print(title, '\n->', 'drift scan' if is_drift else 'tracked spectrum')

## 5. Waterfall — frequency against time

Every record as one row. In kelvin the colour is antenna temperature; in raw
counts it spans orders of magnitude across the band, so plot that in dB (the
log of a temperature would throw the calibration away, so only counts get it).

In [ ]:
if calibrated:
    img, clabel = spectra, 'Antenna temperature (K)'
else:
    img, clabel = 10*np.log10(np.maximum(spectra, 1e-30)), 'Power (dB)'

t_min = (timestamps - timestamps[0]) / 60.0
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(img, aspect='auto', origin='lower', cmap='viridis',
               extent=[freq_mhz[0], freq_mhz[-1], t_min[0], t_min[-1]],
               vmin=np.percentile(img, 5), vmax=np.percentile(img, 95))
fig.colorbar(im, ax=ax, label=clabel)
ax.axvline(1420.405752, color='w', ls='--', lw=0.8, alpha=0.7)
ax.set_xlabel('Frequency (MHz)'); ax.set_ylabel('Minutes from start')
ax.set_title(f'Waterfall - {title}')
plt.tight_layout(); plt.show()

## 6. Mean spectrum, on a velocity axis

Average the **spectra**, never the dB — a mean of logarithms is not the log
of the mean. The x-axis converts frequency to velocity through the Doppler
shift of the 21 cm rest line, 1420.405752 MHz. This is **topocentric** (as
observed); the scheduler's `observation_plot` puts it on the LSR frame, which
is what you compare to published data.

In [ ]:
F_HI = 1420.405752e6
C = 299792.458  # km/s
v_kms = C * (F_HI - freq_hz) / F_HI
mean_spec = spectra.mean(axis=0)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(v_kms, mean_spec, lw=1)
ax.axvline(0, color='r', ls='--', lw=0.8, alpha=0.6, label='H I rest')
ax.set_xlabel('Topocentric velocity (km/s)')
ax.set_ylabel('Antenna temperature (K)' if calibrated else 'Counts')
ax.set_title(f'Mean spectrum - {title}')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Drift curve — band power against time

For a **drift scan** the dish is parked and the sky drifts through the beam,
so the story is in *total power versus time*: the source rises to a peak as it
crosses the beam. (For a tracked spectrum this is meant to be flat — the
interest there is the line, above.) The band power is the median across the H I
channels, robust to the LO spike and narrow interference.

In [ ]:
band_power = np.median(spectra, axis=1)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_min, band_power, lw=1)
ax.set_xlabel('Minutes from start')
ax.set_ylabel('Band-median temperature (K)' if calibrated else 'Band-median counts')
ax.set_title(('Drift curve' if is_drift else 'Band power (tracked - expect flat)') + f' - {title}')
ax.grid(alpha=0.3)
# Mark the recorded beam crossing, if the scheduler computed one.
if 'drift_crossing_time' in attrs:
    try:
        from datetime import datetime, timezone
        tc = datetime.strptime(str(attrs['drift_crossing_time']), '%Y-%m-%d %H:%M:%S')
        tc = tc.replace(tzinfo=timezone.utc).timestamp()
        ax.axvline((tc - timestamps[0]) / 60, color='r', ls='--', lw=0.9, label='beam crossing')
        ax.legend()
    except Exception:
        pass
plt.tight_layout(); plt.show()

## 8. Recovering raw counts (optional)

Kelvin is written reversibly: the bandpass, gain and T_sys all travel in the
file. Undo them to get the counts the SDR produced — needed if you want to
re-reduce with a better bandpass or gain than the one in force at the time.
Fitting a new gain from spectra a gain has already been applied to would just
return unity, so start from counts.

In [ ]:
if calibrated:
    raw = ((spectra + attrs['applied_t_sys_k'])
           * attrs['applied_gain_counts_per_k']
           * hf['bandpass_correction'][:])
    print('recovered raw counts, shape', raw.shape,
          '  mean', f'{raw.mean():.4g}')
else:
    print('already raw counts - nothing to undo')

hf.close()